# Erion Ember x Qwen 3.5 Playground

<a href="https://colab.research.google.com/github/EricNguyen1206/erion-ember/blob/main/docs/templates/erion_ember_qwen35_playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook allows you to test the semantic caching capabilities of Erion Ember using the Qwen3.5-0.8B model.

In [ ]:
# @title (Optional) Mount Google Drive
from google.colab import drive
try:
    drive.mount('/content/drive')
except:
    print("Drive not mounted. Skipping.")

In [3]:
# @title Install Dependencies & Setup Server
# 1. Install Python packages using current python environment
import sys
import subprocess
print("📦 Installing required Python packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "requests", "ipywidgets"])

# 2. Setup Erion Ember Server
import os
import time
import requests
import urllib.request
import platform
import shutil

def setup_server():
    if os.path.exists("erion-ember"):
        os.remove("erion-ember")
    
    is_linux = platform.system() == 'Linux'
    if is_linux:
        try:
            print("🔍 Downloading pre-compiled binary for Linux...")
            urllib.request.urlretrieve("https://github.com/EricNguyen1206/erion-ember/releases/latest/download/erion-ember", "erion-ember")
        except Exception as e:
            print(f"⚠️ Download failed: {e}")
    
    if not os.path.exists("erion-ember"):
        print("🔨 Binary not available. Building from source...")
        if not shutil.which('go'):
            print("⚠️ Go compiler not found.")
            if is_linux:
                print("📥 Attempting to install Go via apt-get...")
                subprocess.check_call("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y golang-go -qq", shell=True)
            else:
                raise RuntimeError("❌ Go is required to build from source. Please install Go (https://go.dev/doc/install).")
        
        if os.path.exists("temp_repo"):
            shutil.rmtree("temp_repo")
        print("📥 Cloning repository...")
        subprocess.check_call(["git", "clone", "--quiet", "https://github.com/EricNguyen1206/erion-ember.git", "temp_repo"])
        print("⚙️ Building Erion Ember...")
        subprocess.check_call(["go", "build", "-ldflags=-s -w", "-o", "../erion-ember", "./cmd/server/"], cwd="temp_repo")
        shutil.rmtree("temp_repo")
    
    if not os.path.exists("erion-ember"):
        raise RuntimeError("❌ Failed to setup erion-ember server.")
        
    os.chmod("erion-ember", 0o755)
    print("🚀 Starting Erion Ember server...")
    
    # Kill existing process on port 8080 if running
    if platform.system() == 'Linux':
        os.system("fuser -k 8080/tcp >/dev/null 2>&1 || true")
    else:
        os.system("lsof -ti:8080 | xargs kill -9 >/dev/null 2>&1 || true")
        
    time.sleep(1)
    subprocess.Popen(["./erion-ember"], stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
    
    # Health check
    for i in range(20):
        try:
            if requests.get("http://localhost:8080/health").status_code == 200:
                print("✅ Erion Ember is ready at http://localhost:8080")
                return
        except:
            pass
        time.sleep(1)
    print("❌ Server failed health check. Check server.log")

setup_server()

In [4]:
# @title Load Qwen3.5-0.8B
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Note: Update to Qwen/Qwen3.5-0.8B-Chat or Base once public
model_id = "Qwen/Qwen2.5-0.8B-Instruct"
print(f"📥 Loading model: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype="auto", 
    device_map="auto", 
    trust_remote_code=True
)

def generate_response(prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(f"✅ Model {model_id} loaded successfully!")

ModuleNotFoundError: No module named 'transformers'

In [ ]:
# @title Interactive Playground
import ipywidgets as widgets
from IPython.display import display, clear_output

prompt1_input = widgets.Textarea(placeholder='Enter initial prompt to cache...', description='Prompt 1:', layout={'width': '80%','height': '100px'})
prompt2_input = widgets.Textarea(placeholder='Enter similar prompt to test cache...', description='Prompt 2:', layout={'width': '80%','height': '100px'})
threshold_slider = widgets.FloatSlider(value=0.85, min=0.0, max=1.0, step=0.01, description='Similarity:', layout={'width': '80%'})
cache_btn = widgets.Button(description="Generate & Cache", button_style='info', layout={'width': '200px'})
test_btn = widgets.Button(description="Query Cache", button_style='success', layout={'width': '200px'})
output_area = widgets.Output()

def on_cache_click(b):
    with output_area:
        clear_output()
        prompt = prompt1_input.value.strip()
        if not prompt: return print("⚠️ Please enter a prompt in Box 1.")
        
        print(f"🤖 Generating response with LLM...")
        response = generate_response(prompt)
        
        print(f"💾 Storing in Erion Ember...")
        res = requests.post("http://localhost:8080/v1/cache/set", json={"prompt": prompt, "response": response})
        if res.status_code == 200:
            token_count = len(tokenizer.encode(prompt + response))
            print(f"✅ Success! Entry stored with ID: {res.json().get('id')}")
            print(f"📊 Total Tokens Indexable: {token_count}")
            print(f"📝 Response: {response[:300]}...")
        else:
            print(f"❌ Cache storage failed: {res.text}")

def on_test_click(b):
    with output_area:
        clear_output()
        prompt = prompt2_input.value.strip()
        if not prompt: return print("⚠️ Please enter a prompt in Box 2.")
        
        print(f"⚡ Querying Semantic Cache...")
        start = time.time()
        try:
            res = requests.post("http://localhost:8080/v1/cache/get", json={
                "prompt": prompt, 
                "similarity_threshold": threshold_slider.value
            })
        except Exception as e:
            return print(f"❌ Server Error: {e}")
        
        latency = (time.time() - start) * 1000
        data = res.json()
        
        if data.get("hit"):
            tokens = len(tokenizer.encode(data['response']))
            print(f"✅ CACHE HIT!")
            print(f"🔗 Similarity: {data['similarity']:.4f}")
            print(f"⏱️ Latency: {latency:.2f}ms")
            print(f"💰 Tokens Saved: {tokens}")
            print(f"📝 Cached Response: {data['response']}")
        else:
            print(f"❌ CACHE MISS (Similarity below {threshold_slider.value})")
            print(f"⏱️ Latency: {latency:.2f}ms")
            print("🤖 Generating new response from scratch...")
            print(f"📝 Result: {generate_response(prompt)}")

cache_btn.on_click(on_cache_click)
test_btn.on_click(on_test_click)

display(widgets.HTML("<h3>1. Populate Cache</h3>"), prompt1_input, cache_btn, 
        widgets.HTML("<h3>2. Test Similarity Match</h3>"), prompt2_input, threshold_slider, test_btn, 
        widgets.HTML("<hr>"), output_area)